# Résumé de l'Architecture du Modèle d'Image Captioning

- **Type** : Encoder-Decoder CNN-RNN
- **Objectif** : Générer des légendes à partir d’images (Flickr30k)
- **Framework** : PyTorch

## Composants

### Encodeur CNN
- **Base** : InceptionV3 pré-entraîné
- **Sortie** : Vecteur `[batch_size, embed_size]` (ex. 256)
- **Détails** : Couche `fc` modifiée, ReLU, dropout (0.5)

### Décodeur RNN
- **Type** : LSTM
- **Entrée** : Caractéristiques + légendes
- **Sortie** : `[seq_len, batch_size, vocab_size]` (ex. 7666)
- **Détails** : Embedding, LSTM, linéaire, dropout (0.5)

### CNNtoRNN
- **Rôle** : Combine encodeur et décodeur
- **Inférence** : Greedy decoding (max 50 mots)

## Données
- **Images** : 299x299, normalisées
- **Légendes** : Tokenisées (SpaCy), paddées

## Entraînement
- **Perte** : CrossEntropy (ignore `<PAD>`)
- **Optimiseur** : Adam (3e-4)
- **Batch** : 32

### `Importation des bibliothèques nécessaires`

In [18]:
import os
import pandas as pd
import spacy
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
import torchvision.models as models
from torchvision.models.inception import Inception_V3_Weights
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.tensorboard import SummaryWriter
from PIL import Image
from tqdm import tqdm

### `Chargement du modèle de langage anglais avec spaCy`

In [19]:
spacy_eng = spacy.load("en_core_web_sm")

/usr/local/lib/python3.10/dist-packages/spacy/util.py:1740: UserWarning: [W111] Jupyter notebook detected: if using `prefer_gpu()` or `require_gpu()`, include it in the same cell right before `spacy.load()` to ensure that the model is loaded on the correct device. More information: http://spacy.io/usage/v3#jupyter-notebook-gpu
  warnings.warn(Warnings.W111)


### `Définition de la classe Vocabulary pour le traitement du texte`


In [20]:
class Vocabulary:
    def __init__(self, freq_threshold):
        self.itos = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>", 3: "<UNK>"}
        self.stoi = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "<UNK>": 3}
        self.freq_threshold = freq_threshold

    def __len__(self):
        return len(self.itos)

    @staticmethod
    def tokenizer_eng(text):
        return [tok.text.lower() for tok in spacy_eng.tokenizer(text)]

    def build_vocabulary(self, sentence_list):
        frequencies = {}
        idx = 4
        for sentence in sentence_list:
            for word in self.tokenizer_eng(sentence):
                frequencies[word] = frequencies.get(word, 0) + 1
                if frequencies[word] == self.freq_threshold:
                    self.stoi[word] = idx
                    self.itos[idx] = word
                    idx += 1

    def numericalize(self, text):
        tokenized_text = self.tokenizer_eng(text)
        return [self.stoi.get(token, self.stoi["<UNK>"]) for token in tokenized_text]


### `Définition de la classe FlickrDataset pour charger les images et leurs légendes`

In [21]:
class FlickrDataset(Dataset):
    def __init__(self, root_dir, captions_file, transform=None, freq_threshold=5):
        self.root_dir = root_dir
        self.transform = transform
        if not os.path.exists(captions_file):
            raise FileNotFoundError(f"Captions file not found: {captions_file}")

        self.df = pd.read_csv(captions_file, sep=",")
        self.df.columns = ["image", "comment_number", "caption"]

        print("Premières lignes de captions.txt :")
        print(self.df.head(10))

        self.imgs = self.df["image"]
        self.captions = self.df["caption"]

        invalid_imgs = [img for img in self.imgs if not (isinstance(img, str) and img.endswith('.jpg'))]
        if invalid_imgs:
            print(f"Noms d'images invalides trouvés : {invalid_imgs[:10]}")
            raise ValueError(f"Certains noms d'images dans captions.txt sont invalides : {invalid_imgs[:5]}")

        self.vocab = Vocabulary(freq_threshold)
        self.vocab.build_vocabulary(self.captions.tolist())

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        caption = self.captions[index]
        img_id = self.imgs[index]
        img_path = os.path.join(self.root_dir, img_id)
        if not os.path.exists(img_path):
            raise FileNotFoundError(f"Image not found: {img_path}")
        img = Image.open(img_path).convert("RGB")

        if self.transform is not None:
            img = self.transform(img)

        numericalized_caption = [self.vocab.stoi["<SOS>"]]
        numericalized_caption += self.vocab.numericalize(caption)
        numericalized_caption.append(self.vocab.stoi["<EOS>"])

        return img, torch.tensor(numericalized_caption)

### `Définition de la classe MyCollate pour le padding des séquences et la préparation du batch`


In [22]:
class MyCollate:
    def __init__(self, pad_idx):
        self.pad_idx = pad_idx

    def __call__(self, batch):
        imgs = torch.cat([item[0].unsqueeze(0) for item in batch], dim=0)
        targets = pad_sequence([item[1] for item in batch], batch_first=False, padding_value=self.pad_idx)
        return imgs, targets

### `Définition de la fonction get_loader pour créer un DataLoader avec padding et transformations`

In [23]:
def get_loader(
    root_folder,
    annotation_file,
    transform,
    batch_size=32,
    num_workers=2,
    shuffle=True,
    pin_memory=True,
):
    dataset = FlickrDataset(root_folder, annotation_file, transform=transform)
    pad_idx = dataset.vocab.stoi["<PAD>"]

    loader = DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        num_workers=num_workers,
        shuffle=shuffle,
        pin_memory=pin_memory,
        collate_fn=MyCollate(pad_idx=pad_idx),
    )

    return loader, dataset

### `Application des transformations et chargement des données avec DataLoader`


In [24]:
transform = transforms.Compose([
    transforms.Resize((356, 356)),
    transforms.RandomCrop((299, 299)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

loader, dataset = get_loader(
    "/kaggle/input/flickr30k/flickr30k_images",
    "/kaggle/input/flickr30k/captions.txt",
    transform=transform,
    batch_size=2
)

for idx, (imgs, captions) in enumerate(loader):
    print(f"Batch {idx}:")
    print(f"Images shape: {imgs.shape}")
    print(f"Captions shape: {captions.shape}")
    break

Premières lignes de captions.txt :
            image  comment_number  \
0  1000092795.jpg               0   
1  1000092795.jpg               1   
2  1000092795.jpg               2   
3  1000092795.jpg               3   
4  1000092795.jpg               4   
5    10002456.jpg               0   
6    10002456.jpg               1   
7    10002456.jpg               2   
8    10002456.jpg               3   
9    10002456.jpg               4   

                                             caption  
0  Two young guys with shaggy hair look at their ...  
1  Two young  White males are outside near many b...  
2   Two men in green shirts are standing in a yard .  
3       A man in a blue shirt standing in a garden .  
4            Two friends enjoy time spent together .  
5  Several men in hard hats are operating a giant...  
6  Workers look down from up above on a piece of ...  
7   Two men working on a machine wearing hard hats .  
8              Four men on top of a tall structure .  
9      

### `Définition des classes EncoderCNN, DecoderRNN et CNNtoRNN pour l'architecture du modèle d'image captioning`

In [25]:
class EncoderCNN(nn.Module):
    def __init__(self, embed_size, train_CNN=False):
        super(EncoderCNN, self).__init__()
        self.train_CNN = train_CNN
        self.inception = models.inception_v3(weights=Inception_V3_Weights.IMAGENET1K_V1)
        self.inception.fc = nn.Linear(self.inception.fc.in_features, embed_size)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)

        if not self.train_CNN:
            for param in self.inception.parameters():
                param.requires_grad = False
            for param in self.inception.fc.parameters():
                param.requires_grad = True

    def forward(self, images):
        outputs = self.inception(images)
        features = outputs if not isinstance(outputs, tuple) else outputs[0]
        return self.dropout(self.relu(features))


class DecoderRNN(nn.Module):
    def __init__(self, embed_size, hidden_size, vocab_size, num_layers):
        super(DecoderRNN, self).__init__()
        self.embed = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers, batch_first=False)
        self.linear = nn.Linear(hidden_size, vocab_size)
        self.dropout = nn.Dropout(0.5)

    def forward(self, features, captions):
        embeddings = self.dropout(self.embed(captions))
        embeddings = torch.cat((features.unsqueeze(0), embeddings), dim=0)
        hiddens, _ = self.lstm(embeddings)
        outputs = self.linear(hiddens)
        return outputs


class CNNtoRNN(nn.Module):
    def __init__(self, embed_size, hidden_size, vocab_size, num_layers):
        super(CNNtoRNN, self).__init__()
        self.encoderCNN = EncoderCNN(embed_size)
        self.decoderRNN = DecoderRNN(embed_size, hidden_size, vocab_size, num_layers)

    def forward(self, images, captions):
        features = self.encoderCNN(images)
        outputs = self.decoderRNN(features, captions)
        return outputs

    def caption_image(self, image, vocabulary, max_length=50):
        result_caption = []
        with torch.no_grad():
            x = self.encoderCNN(image).unsqueeze(0)
            states = None

            for _ in range(max_length):
                hiddens, states = self.decoderRNN.lstm(x, states)
                output = self.decoderRNN.linear(hiddens.squeeze(0))
                predicted = output.argmax(1)
                result_caption.append(predicted.item())
                x = self.decoderRNN.embed(predicted).unsqueeze(0)
                if vocabulary.itos[predicted.item()] == "<EOS>":
                    break

        return [vocabulary.itos[idx] for idx in result_caption]

### `Fonction d'entraînement du modèle pour l'image captioning`


In [26]:
def train():
    transform = transforms.Compose([
        transforms.Resize((356, 356)),
        transforms.RandomCrop((299, 299)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    ])

    train_loader, dataset = get_loader(
        root_folder="/kaggle/input/flickr30k/flickr30k_images",
        annotation_file="/kaggle/input/flickr30k/captions.txt",
        transform=transform,
        num_workers=2,
    )

    torch.backends.cudnn.benchmark = True
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    load_model = False
    save_model = True
    train_CNN = False

    embed_size = 256
    hidden_size = 256
    vocab_size = len(dataset.vocab)
    num_layers = 1
    learning_rate = 3e-3
    num_epochs = 10

    writer = SummaryWriter("runs/flickr30k")
    step = 0

    model = CNNtoRNN(embed_size, hidden_size, vocab_size, num_layers).to(device)
    criterion = nn.CrossEntropyLoss(ignore_index=dataset.vocab.stoi["<PAD>"])
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    for name, param in model.encoderCNN.inception.named_parameters():
        param.requires_grad = ("fc" in name) or train_CNN

    if load_model:
        step = load_checkpoint(torch.load("my_checkpoint.pth.tar"), model, optimizer)

    model.train()

    for epoch in range(num_epochs):
        print(f"Epoch [{epoch+1}/{num_epochs}]")
        total_loss = 0

        for idx, (imgs, captions) in tqdm(enumerate(train_loader), total=len(train_loader), leave=False):
            imgs = imgs.to(device)
            captions = captions.to(device)

            outputs = model(imgs, captions[:-1])
            targets = captions[1:]

            outputs = outputs[1:]

            assert outputs.shape[0] == targets.shape[0], \
                f"Mismatch in sequence length: outputs {outputs.shape}, targets {targets.shape}"

            loss = criterion(outputs.reshape(-1, outputs.shape[2]), targets.reshape(-1))

            writer.add_scalar("Training loss", loss.item(), global_step=step)
            step += 1

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"Average Loss: {avg_loss:.4f}")

        if save_model:
            checkpoint = {
                "state_dict": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "step": step,
            }
            torch.save(checkpoint, "my_checkpoint.pth.tar")
            print("Checkpoint saved.")

    writer.close()

In [27]:
train()

Premières lignes de captions.txt :
            image  comment_number  \
0  1000092795.jpg               0   
1  1000092795.jpg               1   
2  1000092795.jpg               2   
3  1000092795.jpg               3   
4  1000092795.jpg               4   
5    10002456.jpg               0   
6    10002456.jpg               1   
7    10002456.jpg               2   
8    10002456.jpg               3   
9    10002456.jpg               4   

                                             caption  
0  Two young guys with shaggy hair look at their ...  
1  Two young  White males are outside near many b...  
2   Two men in green shirts are standing in a yard .  
3       A man in a blue shirt standing in a garden .  
4            Two friends enjoy time spent together .  
5  Several men in hard hats are operating a giant...  
6  Workers look down from up above on a piece of ...  
7   Two men working on a machine wearing hard hats .  
8              Four men on top of a tall structure .  
9      

Average Loss: 3.5390
Checkpoint saved.
Epoch [2/10]


Average Loss: 3.2333
Checkpoint saved.
Epoch [3/10]


Average Loss: 3.1455
Checkpoint saved.
Epoch [4/10]


Average Loss: 3.0934
Checkpoint saved.
Epoch [5/10]


...

Average Loss: 3.0304
Checkpoint saved.
Epoch [7/10]


Average Loss: 3.0100
Checkpoint saved.
Epoch [8/10]


Average Loss: 2.9941
Checkpoint saved.
Epoch [9/10]


Average Loss: 2.9822
Checkpoint saved.
Epoch [10/10]


Average Loss: 2.9707
Checkpoint saved.
